# Speech Enhancement Demo — Audio Analysis (Colab)

This notebook loads **pre-computed enhanced audio** from Google Drive and provides
a detailed comparison of all 4 conditioning variants:

| # | Variant | File |
|---|---------|------|
| 1 | Clean reference | `1_clean.wav` |
| 2 | Noisy input (5 dB SNR) | `2_noisy_snr5dB.wav` |
| 3 | Enhanced — No conditioning | `3_enhanced_none.wav` |
| 4 | Enhanced — Last layer | `4_enhanced_last_layer.wav` |
| 5 | Enhanced — Static multi-layer | `5_enhanced_multi_layer.wav` |
| 6 | Enhanced — Time-dep. multi-layer | `6_enhanced_multi_layer_time.wav` |

**Analyses performed:**
1. Audio playback (listen in browser)
2. Mel-spectrogram comparison
3. Spectral difference (residual vs. clean)
4. Waveform overlay
5. Zoomed waveform segment
6. PESQ & STOI scores
7. Frequency band energy profiles

## 1. Setup — Mount Google Drive & Install Dependencies

Mount your Google Drive so we can access the pre-computed outputs directly.
Then install the audio analysis libraries.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q librosa soundfile pesq pystoi matplotlib numpy

## 2. Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display
import IPython.display as ipd

try:
    from pesq import pesq
    HAS_PESQ = True
except ImportError:
    HAS_PESQ = False
    print("pesq not installed — skipping PESQ computation")

try:
    from pystoi import stoi
    HAS_STOI = True
except ImportError:
    HAS_STOI = False
    print("pystoi not installed — skipping STOI computation")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 4)

SR = 16_000  # all audio is 16 kHz
print("Imports OK")

## 3. Configuration

Set `SAMPLE_DIR` to the folder on your Google Drive that contains the 6 wav files
for the sample you want to analyse.

The outputs are stored at:
`speech_enhancement_outputs/audio_samples/<sample_name>/`

Each folder contains exactly 6 files:
`1_clean.wav`, `2_noisy_snr5dB.wav`, `3_enhanced_none.wav`,
`4_enhanced_last_layer.wav`, `5_enhanced_multi_layer.wav`,
`6_enhanced_multi_layer_time.wav`.

In [ ]:
# ---- CHANGE THIS PATH IF NEEDED ----
DRIVE_ROOT = "/content/drive/MyDrive"
OUTPUTS_DIR = os.path.join(
    DRIVE_ROOT,
    "speech_enhancement_outputs",
    "audio_samples",
)

# Pick a sample folder (list available ones first)
samples = sorted([
    d for d in os.listdir(OUTPUTS_DIR)
    if os.path.isdir(os.path.join(OUTPUTS_DIR, d))
])
print(f"Available samples ({len(samples)}):")
for s in samples[:20]:
    print(f"  {s}")
if len(samples) > 20:
    print(f"  ... and {len(samples) - 20} more")

## 4. Select Sample & Load Audio

Pick a sample from the list above (or keep the default).

In [ ]:
# ---- SELECT YOUR SAMPLE ----
SAMPLE_NAME = "LibriSpeech_dev-clean_5694_64038_5694-64038-0010"
# -----------------------------

SAMPLE_DIR = os.path.join(OUTPUTS_DIR, SAMPLE_NAME)
assert os.path.isdir(SAMPLE_DIR), f"Not found: {SAMPLE_DIR}"

# File mapping
FILES = {
    "clean":            os.path.join(SAMPLE_DIR, "1_clean.wav"),
    "noisy":            os.path.join(SAMPLE_DIR, "2_noisy_snr5dB.wav"),
    "none":             os.path.join(SAMPLE_DIR, "3_enhanced_none.wav"),
    "last_layer":       os.path.join(SAMPLE_DIR, "4_enhanced_last_layer.wav"),
    "multi_layer":      os.path.join(SAMPLE_DIR, "5_enhanced_multi_layer.wav"),
    "multi_layer_time": os.path.join(SAMPLE_DIR, "6_enhanced_multi_layer_time.wav"),
}

# Load all
wavs = {}
for key, path in FILES.items():
    if os.path.exists(path):
        wavs[key], _ = librosa.load(path, sr=SR)
        print(f"  {key:20s}: {len(wavs[key])/SR:.2f}s  ({path.split('/')[-1]})")
    else:
        print(f"  WARNING — not found: {path}")

LABELS = {
    "clean": "Clean (ref)",
    "noisy": "Noisy (5 dB)",
    "none": "No Cond.",
    "last_layer": "Last Layer",
    "multi_layer": "Static ML",
    "multi_layer_time": "Time-dep. ML",
}

CONDITIONS = ["none", "last_layer", "multi_layer", "multi_layer_time"]
print(f"\nLoaded {len(wavs)} / 6 files from: {SAMPLE_NAME}")

## 5. Audio Playback

Listen to each variant directly in the browser. Compare noise suppression
and speech naturalness across the 4 enhanced versions.

In [ ]:
for key in ["clean", "noisy"] + CONDITIONS:
    if key in wavs:
        print(f"\n{LABELS[key]}")
        ipd.display(ipd.Audio(wavs[key], rate=SR))

## 6. Mel-Spectrogram Comparison

Side-by-side log-Mel spectrograms. Look for:
- Noise floor reduction (dark areas becoming darker)
- Harmonic structure preservation (horizontal lines)
- Artefact introduction (unexpected bright patches)

In [ ]:
def compute_mel(wav, sr=SR):
    mel = librosa.feature.melspectrogram(
        y=wav, sr=sr, n_fft=1024, hop_length=256, n_mels=80
    )
    return librosa.power_to_db(mel, ref=np.max)

panel_keys = ["clean", "noisy"] + [c for c in CONDITIONS if c in wavs]
n = len(panel_keys)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 3.5))
if n == 1:
    axes = [axes]

for ax, key in zip(axes, panel_keys):
    mel_db = compute_mel(wavs[key])
    img = librosa.display.specshow(
        mel_db, sr=SR, hop_length=256, x_axis="time", y_axis="mel", ax=ax
    )
    ax.set_title(LABELS[key], fontsize=9)
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

fig.suptitle(f"Mel-Spectrogram Comparison — {SAMPLE_NAME}", fontsize=11)
plt.tight_layout()
plt.show()

## 7. Spectral Difference (Enhanced − Clean)

The **absolute mel-spectrogram residual** between each variant and clean.
Smaller (darker) residuals = better reconstruction.

In [ ]:
clean_mel = compute_mel(wavs["clean"])
noisy_mel = compute_mel(wavs["noisy"])

# Noisy residual
min_t = min(clean_mel.shape[1], noisy_mel.shape[1])
noisy_diff = np.abs(noisy_mel[:, :min_t] - clean_mel[:, :min_t])

diff_panels = [("Noisy − Clean", noisy_diff)]
for ct in CONDITIONS:
    if ct in wavs:
        enh_mel = compute_mel(wavs[ct])
        mt = min(clean_mel.shape[1], enh_mel.shape[1])
        diff_panels.append((f"{LABELS[ct]} − Clean", np.abs(enh_mel[:, :mt] - clean_mel[:, :mt])))

n = len(diff_panels)
fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 3.5))

for ax, (title, diff) in zip(axes, diff_panels):
    img = librosa.display.specshow(
        diff, sr=SR, hop_length=256, x_axis="time", y_axis="mel", ax=ax
    )
    ax.set_title(title, fontsize=9)
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

fig.suptitle("Spectral Residual |Enhanced − Clean|", fontsize=11)
plt.tight_layout()
plt.show()

## 8. Waveform Comparison

Full-length time-domain waveforms stacked vertically.
Compare the overall envelope and noise floor.

In [ ]:
plot_keys = ["clean", "noisy"] + [c for c in CONDITIONS if c in wavs]
n = len(plot_keys)
fig, axes = plt.subplots(n, 1, figsize=(14, 2.2 * n), sharex=True)

for ax, key in zip(axes, plot_keys):
    wav = wavs[key]
    t = np.arange(len(wav)) / SR
    ax.plot(t, wav, linewidth=0.3, color="steelblue")
    ax.set_ylabel(LABELS[key], fontsize=8, rotation=0, labelpad=80, va="center")
    ax.set_xlim(0, t[-1])
    ax.set_ylim(-1, 1)
    ax.tick_params(labelsize=7)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Waveform Comparison", fontsize=11)
plt.tight_layout()
plt.show()

## 9. Zoomed Waveform (0.5 s segment)

Zoom into a short segment to see fine waveform detail — harmonic structure,
transient preservation, and residual noise.

In [ ]:
start_sec = 1.0
dur_sec = 0.5
s0, s1 = int(start_sec * SR), int((start_sec + dur_sec) * SR)

fig, axes = plt.subplots(len(plot_keys), 1, figsize=(14, 2.0 * len(plot_keys)), sharex=True)

for ax, key in zip(axes, plot_keys):
    wav = wavs[key]
    seg = wav[s0:s1] if len(wav) > s1 else wav[s0:]
    t = np.arange(len(seg)) / SR + start_sec
    ax.plot(t, seg, linewidth=0.6, color="steelblue")
    ax.set_ylabel(LABELS[key], fontsize=8, rotation=0, labelpad=80, va="center")
    ax.tick_params(labelsize=7)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(f"Zoomed Waveform ({start_sec:.1f}–{start_sec+dur_sec:.1f} s)", fontsize=11)
plt.tight_layout()
plt.show()

## 10. Per-Sample Metrics (PESQ, STOI) & FAD

**PESQ** (Perceptual Evaluation of Speech Quality, higher = better): measures
perceived speech quality on a 1–4.5 scale.

**STOI** (Short-Time Objective Intelligibility, higher = better): measures
how intelligible the speech is, range 0–1.

**FAD** (Fréchet Audio Distance, lower = better): measures distributional
similarity between generated and reference audio in VGGish embedding space.
FAD is a *batch* metric computed over the full test set (270 samples), so we
show the pre-computed values from evaluation.

In [ ]:
clean = wavs["clean"]

# --- Per-sample PESQ & STOI ---
print(f"{'Variant':<20s}  {'PESQ':>8s}  {'STOI':>8s}")
print(f"{'-'*20}  {'-'*8}  {'-'*8}")

for key in ["noisy"] + CONDITIONS:
    if key not in wavs:
        continue
    w = wavs[key]
    ml = min(len(clean), len(w))
    p = pesq(SR, clean[:ml], w[:ml], "wb") if HAS_PESQ else float("nan")
    s = stoi(clean[:ml], w[:ml], SR, extended=False) if HAS_STOI else float("nan")
    print(f"{LABELS[key]:<20s}  {p:8.4f}  {s:8.4f}")

# --- FAD (batch metric, pre-computed over full 270-sample test set) ---
FAD_SCORES = {
    "none":             2.9774,
    "last_layer":       2.6997,
    "multi_layer":      2.3857,
    "multi_layer_time": 2.3456,
}

print("\n--- FAD (full test set, 270 samples) ---")
print(f"{'Variant':<20s}  {'FAD':>8s}")
print(f"{'-'*20}  {'-'*8}")
for ct in CONDITIONS:
    print(f"{LABELS[ct]:<20s}  {FAD_SCORES[ct]:8.4f}")

## 11. Summary

Key observations from this sample:

1. **No conditioning** (Exp 1) removes some noise but introduces artefacts and loses speech detail.
2. **Last-layer conditioning** (Exp 2) improves clarity but still misses fine-grained acoustic structure.
3. **Static multi-layer** (Exp 3) captures more spectral detail — the spectral residual is visibly smaller.
4. **Time-dependent multi-layer** (Exp 4) produces the closest match to clean speech in both spectrogram and waveform views.

The progression confirms our core finding: **multi-layer tokenizer features provide
complementary information** that progressively improves enhancement quality.

| Metric | No Cond. | Last Layer | Static ML | Time-dep. ML |
|--------|----------|------------|-----------|-------------- |
| PESQ ↑ | 1.6048   | 1.6499     | 1.6868    | **1.6986**   |
| STOI ↑ | 0.8527   | 0.8589     | 0.8642    | **0.8647**   |
| FAD ↓  | 2.9774   | 2.6997     | 2.3857    | **2.3456**   |